In [0]:
import requests
import json
from datetime import datetime, timezone
from pyspark.sql.functions import col, parse_json

In [0]:
# Define destination path inside Unity Catalog Volume
write_path = "the_data_masons.landing"

In [0]:
# Set API url
url = 'https://api.eia.gov/v2/electricity/rto/region-sub-ba-data/data/'
api_key = "52QFUWuo1qHOtc3jnd40P6cTGznr1fLmpltudeq6"
time_period = "2026-01-01T00"

# Define file prefix
table_name = "raw_demand_by_zone"

In [0]:
def build_params(tracker):
    print("Building params")
    return {
        "api_key": api_key,
        "frequency":"hourly",
        "data": ["value"],
        "start":time_period,
        "end":time_period,
        "facets": {
            "parent": [
                "NYIS"
            ]
        },
        "sort": [
            {
                "column": "period",
                "direction": "asc"
            },
            {
                "column": "parent",
                "direction": "asc"
            },
            {
                "column": "subba",
                "direction": "asc"
            }
        ],
        "offset": tracker.offset,
        "length": 5000
    }
    

In [0]:
def fetch_response(url, params):
    print("Fetching data")
    return requests.get(url, params=params)

In [0]:
def parse_response(response):
    print("Parsing response")

    # check response status
    response.raise_for_status()

    return response.json()

In [0]:
def construct_record(timestamp, response):
    print("Constructing record")
    return {
        "request_timestamp": timestamp,
        "status_code": response.status_code,
        "status": response.reason,
        "raw_response" : json.dumps(response.json()),
        "offset":"",
        "page":"",
    }

In [0]:
def create_dataframe(buffer):
    print("Creating dataframe from buffer")
    return (spark.createDataFrame(buffer)
    .withColumn("raw_response", parse_json(col("raw_response"))))

In [0]:
# create a Delta Table buffer array
landing_buffer = []
tracker = {
    "offset" : 0,
    "page" : 0
}
has_more_data = TRUE
# loop and append response record to the buffer
for i in range(5):

    params = build_params(tracker)

    # get request time
    timestamp = datetime.now(timezone.utc)

    # get request resonse
    response = fetch_response(url, params)

    # check for errors
    response.raise_for_status()

    # no errors, then Construct record
    landing_record = construct_record(timestamp, response)

    # append record to buffer
    print(f"Appending record {i} to buffer")
    print(landing_record)   
    landing_buffer.append(landing_record)


In [0]:
# create a dataframe from the buffer array
df_landing = create_dataframe(landing_buffer)
                                              
display(df_landing)

In [0]:
# Save the table to the landing schema
df_landing.write.mode("append").saveAsTable(f"{write_path}.{table_name}")   